In [ ]:
# ── CV Parser (MiMo / Xiaomi): single-pass structured extraction ───────
# Pipeline: PDF --(pymupdf)--> teks bersih --(MiMo)--> JSON sesuai skema
# Output: data/processed_mimo/ (terpisah dari Ollama & Gemini)

from openai import OpenAI
import fitz
import json
import unicodedata
import re
from pathlib import Path

# ── Config ───────────────────────────────────────────────────────────
API_KEY  = "sk-sfoim1txpf2e6ng5eztc4e50996hbtgby6wug8hfrur87few"
BASE_URL = "https://api.xiaomimimo.com/v1"
MODEL    = "mimo-v2.5"   # atau mimo-v2-flash (lebih cepat) / mimo-v2.5-pro (lebih pintar)

client  = OpenAI(api_key=API_KEY, base_url=BASE_URL)
CV_DIR  = Path("../../data/cv")
OUT_DIR = Path("../../data/processed_mimo")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Config OK | model: {MODEL}")

In [ ]:
# ── 1) Ekstraksi teks PDF ─────────────────────────────────────────────

def clean_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u2014", " - ").replace("\u2013", " - ").replace("\u2022", " - ")
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def extract_clean_text(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    full_text = []
    for page in doc:
        data = page.get_text("dict")
        page_text = []
        for block in data["blocks"]:
            if "lines" not in block:
                continue
            for line in block["lines"]:
                line_text = " ".join(span["text"] for span in line["spans"]).strip()
                if line_text:
                    page_text.append(line_text)
        full_text.append(" ".join(page_text))
    doc.close()
    return clean_text(" ".join(full_text))

In [ ]:
# ── 2) Skema JSON + ekstraksi via MiMo ───────────────────────────────
CV_SCHEMA = {
    "type": "object",
    "properties": {
        "personal_info": {
            "type": "object",
            "properties": {
                "name":     {"type": "string"},
                "email":    {"type": "string"},
                "phone":    {"type": "string"},
                "location": {"type": "string"},
                "links":    {"type": "array", "items": {"type": "string"}},
            },
            "required": ["name", "email", "phone", "location", "links"],
        },
        "summary": {"type": "string"},
        "skills": {
            "type": "object",
            "properties": {
                "hard_skills": {"type": "array", "items": {"type": "string"}},
                "soft_skills": {"type": "array", "items": {"type": "string"}},
            },
            "required": ["hard_skills", "soft_skills"],
        },
        "experience": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "company":    {"type": "string"},
                    "role":       {"type": "string"},
                    "location":   {"type": "string"},
                    "start_date": {"type": "string"},
                    "end_date":   {"type": "string"},
                    "is_current": {"type": "boolean"},
                    "summary":    {"type": "string"},
                    "key_skills": {"type": "array", "items": {"type": "string"}},
                },
                "required": ["company", "role", "location", "start_date",
                             "end_date", "is_current", "summary", "key_skills"],
            },
        },
        "education": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "institution":    {"type": "string"},
                    "degree":         {"type": "string"},
                    "field_of_study": {"type": "string"},
                    "start_year":     {"type": "string"},
                    "end_year":       {"type": "string"},
                },
                "required": ["institution", "degree", "field_of_study",
                             "start_year", "end_year"],
            },
        },
        "certifications": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name":   {"type": "string"},
                    "issuer": {"type": "string"},
                    "year":   {"type": "string"},
                },
                "required": ["name", "issuer", "year"],
            },
        },
        "achievements": {"type": "array", "items": {"type": "string"}},
        "languages": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "language":    {"type": "string"},
                    "proficiency": {"type": "string"},
                },
                "required": ["language", "proficiency"],
            },
        },
    },
    "required": ["personal_info", "summary", "skills", "experience",
                 "education", "certifications", "achievements", "languages"],
}

SYSTEM_PROMPT = """You are an expert CV/resume parser for ANY job sector.
Extract the CV into JSON. You MUST use EXACTLY these field names — no variations allowed:
- "personal_info" (NOT personal_information, NOT personalInfo)
- "summary" (NOT profile_summary, NOT about)
- "experience" (NOT work_experience, NOT employment)
- "education", "skills", "certifications", "achievements", "languages"

SKILLS: hard_skills = professional competencies. soft_skills = ONLY if explicitly listed.
DATES: Only output dates EXPLICITLY in CV. Format "YYYY-MM". If absent use "".
"Present"/"Sekarang"/"saat ini" => is_current=true, end_date="".
Output ONLY valid JSON, no explanation."""


def atomize_skills(items):
    out = []
    for s in items:
        if not isinstance(s, str):
            continue
        if ":" in s:
            s = s.split(":", 1)[1]
        for part in re.split(r"[,|]", s):
            p = part.strip(" .;-•")
            if p:
                out.append(p)
    seen, result = set(), []
    for x in out:
        if x.lower() not in seen:
            seen.add(x.lower())
            result.append(x)
    return result


def normalize_cv(data: dict) -> dict:
    """Normalisasi field name jika MiMo pakai nama berbeda dari schema."""
    # personal_info
    if "personal_info" not in data:
        for alt in ["personal_information", "personalInfo", "contact", "personal"]:
            if alt in data:
                data["personal_info"] = data.pop(alt)
                break
    pi = data.get("personal_info", {})
    if isinstance(pi, dict) and "links" not in pi:
        for alt in ["linkedin", "github", "portfolio", "website"]:
            if alt in pi:
                pi["links"] = [pi.pop(alt)]
                break
        else:
            pi["links"] = []

    # summary
    if "summary" not in data:
        for alt in ["profile_summary", "profileSummary", "about", "objective", "profile"]:
            if alt in data:
                data["summary"] = data.pop(alt)
                break

    # experience
    if "experience" not in data:
        for alt in ["work_experience", "workExperience", "employment", "work_history"]:
            if alt in data:
                data["experience"] = data.pop(alt)
                break

    # pastikan field wajib ada
    for field in ["summary", "experience", "education", "certifications", "achievements", "languages"]:
        data.setdefault(field, [] if field != "summary" else "")
    data.setdefault("skills", {"hard_skills": [], "soft_skills": []})
    data.setdefault("personal_info", {"name": "", "email": "", "phone": "", "location": "", "links": []})

    return data


def _year_in_source(date_str, text):
    return (not date_str) or (date_str[:4] in text)


def validate_dates(data, source_text):
    for e in data.get("experience", []):
        if not isinstance(e, dict):
            continue
        for k in ("start_date", "end_date"):
            if e.get(k) and not _year_in_source(e[k], source_text):
                e[k] = ""
        if not e.get("start_date") and not e.get("end_date"):
            e["is_current"] = False
    return data


def extract_cv(cv_text: str) -> dict:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": cv_text},
        ],
        response_format={"type": "json_object"},
        temperature=0,
    )
    data = json.loads(resp.choices[0].message.content)
    data = normalize_cv(data)
    data["skills"]["hard_skills"] = atomize_skills(data["skills"].get("hard_skills", []))
    data["skills"]["soft_skills"] = atomize_skills(data["skills"].get("soft_skills", []))
    for e in data.get("experience", []):
        if isinstance(e, dict):
            e["key_skills"] = atomize_skills(e.get("key_skills", []))
    return validate_dates(data, cv_text)

In [ ]:
# ── 3) Batch: proses SEMUA PDF ────────────────────────────────────────
import time

def extract_cv_with_retry(cv_text, max_retries=4):
    for attempt in range(max_retries):
        try:
            return extract_cv(cv_text)
        except Exception as e:
            msg = str(e)
            if "429" in msg:
                import re as _re
                m = _re.search(r"(\d+)s", msg)
                wait = int(m.group(1)) + 5 if m else 60
                print(f"      rate limit, tunggu {wait}s ...", end=" ", flush=True)
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Gagal setelah semua retry")

pdf_files = sorted(CV_DIR.glob("*.pdf"))
print(f"Ditemukan {len(pdf_files)} CV di {CV_DIR}\n")

results = {}
for i, pdf_path in enumerate(pdf_files, 1):
    print(f"[{i}/{len(pdf_files)}] {pdf_path.name} ...", end=" ", flush=True)
    try:
        cv_text = extract_clean_text(str(pdf_path))
        data    = extract_cv_with_retry(cv_text)
        out_path = OUT_DIR / f"{pdf_path.stem}.json"
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        results[pdf_path.name] = data
        print(f"OK -> {out_path.name} ({len(data.get('experience', []))} pengalaman)")
        time.sleep(2)
    except Exception as e:
        print(f"GAGAL: {e}")

print(f"\nSelesai. {len(results)}/{len(pdf_files)} CV berhasil. JSON di: {OUT_DIR}/")

In [ ]:
# ── 4) Preview hasil CV pertama ───────────────────────────────────────
if results:
    first_name = next(iter(results))
    print(f"Preview: {first_name}\n")
    print(json.dumps(results[first_name], indent=2, ensure_ascii=False))
else:
    print("Belum ada hasil. Jalankan cell batch di atas dulu.")